##### Load data

In [1]:
import pandas as pd
import json

df = pd.read_csv('Environmental Processes.csv', encoding_errors='ignore')
links = df['Link'].tolist()
abs = df['ab_content'].tolist()
hts = df['hi_content'].tolist()
abs = [ab.strip() for ab in abs]
hts = [ht.strip().replace('\n', '') for ht in hts]

link_prefix = 'https://link.springer.com/article/'
links = [link.replace(link_prefix, '') for link in links]

link_to_keywords = {}
with open("EP_keywords.txt", 'r', encoding='utf-8') as f:
    for line in f.readlines():
        items = line.strip().split('\\t')
        link = items[0]
        keywords = items[1].split(";")
        link_to_keywords[link] = keywords

In [2]:
ahts, hats = [], []

for i in range(len(abs)):
    ahts.append(abs[i] + ' ' + hts[i])
    hats.append(hts[i] + ' ' + abs[i])

In [7]:
keywords = []
for link in links:
    keywords.append(link_to_keywords[link])

##### GPT-3.5

In [ ]:
from langchain.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain.schema import HumanMessage, SystemMessage
import json

# Prompt 模板
system_template = SystemMessagePromptTemplate.from_template("""
You are an expert in the academic field of environmental science.                                           
""")

human_template = HumanMessagePromptTemplate.from_template("""
Based on the text below, give the keyword set of the paper in which it is located, 
sort the results from high to low importance, and separate the results with commas. 
Note that only the results are given without any additional explanations:\n\"{text}\"
""")

prompt_template = ChatPromptTemplate.from_messages([system_template, human_template])

In [24]:
from tqdm import tqdm
from langchain.chat_models import ChatOpenAI

my_key = 'sk-BeMJ7hDAK0eIcAI0EhHO7wODq99mtWiGtWLe3yhDofID0GeQ'

llm = ChatOpenAI(model_name="gpt-4o-mini-2024-07-18", temperature=0.2, openai_api_key=my_key, openai_api_base="https://api.chatgptsb.com/v1/")

# 分析函数
def analyze_abstract(abstract_text: str):
    prompt = prompt_template.format_messages(text=abstract_text)
    response = llm(prompt)
    result = response.content
    return result

keywords_pred = []

for text in tqdm(hats):
    result = analyze_abstract(text)
    keywords_pred.append(result)

100%|██████████| 100/100 [14:34<00:00,  8.75s/it]  


In [25]:
new_df = pd.DataFrame()
new_df['link'] = links
new_df['Keywords'] = keywords_pred
new_df.to_excel('data/EP-GPT-HA.xlsx', index=False)

##### Claude3

In [4]:
from tqdm import tqdm
from langchain.chat_models import ChatAnthropic
from langchain.schema import HumanMessage
import warnings
warnings.filterwarnings("ignore")

my_key = 'sk-NHWa9whhit4nTxfP5cWz6HRsKXrzSaWjcmDTJ9NfXKAg26A8'

llm = ChatAnthropic(
    model="claude-3-sonnet-20240229",
    temperature=0.2,
    anthropic_api_key=my_key,
    anthropic_api_base="https://api.chatgptsb.com/v1/"
)

# 分析函数
def analyze_abstract(abstract_text: str):
    prompt = prompt_template.format_messages(text=abstract_text)
    response = llm(prompt)
    result = response.content
    return result

keywords_pred = []

for text in tqdm(hts):
    result = analyze_abstract(text)
    keywords_pred.append(result)

AttributeError: 'Anthropic' object has no attribute 'count_tokens'

In [ ]:
new_df = pd.DataFrame()
new_df['Keywords'] = keywords_pred
new_df.to_excel('ReviewPlus/data/EP-Claude-H.xlsx', index=False)

##### Evaluation

In [ ]:
import re

new_df = pd.read_excel('DATA/EP-GPT-HA.xlsx')
raw_pred_keywords = new_df['Keywords'].tolist()
pred_keywords = []
regex = r'\d. '
for elem in raw_pred_keywords:
    keyword = []
    if ':' in elem:
        pos = elem.rindex(':')
        elem = elem[pos+1:]
        if ',' in elem:
            keyword = elem.split(',')
        elif '\n' in elem:
            keyword = elem.split('\n')
    if ':' not in elem:
        if ',' in elem:
            keyword = elem.split(',')
        elif '\n' in elem:
            keyword = elem.split('\n')

    new_keyword = []
    for i, word in enumerate(keyword):
        word = word.strip()
        if len(word.split(' ')) >=5:
            continue
        if '- ' in word:
            pos = word.index(' ')
            word = word[pos+1:]
        elif re.search(regex, word):
            pos = re.search(regex, word).span()[1]
            word = word[pos:]

        if word != '':
            if word[-1] == '.':
                new_keyword.append(word[:-1])
            else:
                new_keyword.append(word)

    keyword = [word.strip() for word in new_keyword]
    pred_keywords.append(keyword)

In [4]:
import nltk

porter = nltk.PorterStemmer()

def stemmer(raw_sequences):
    stemmed_sequences = []

    for i, words in enumerate(raw_sequences):
        new_words = []
        for word in words:
            if type(word) == list:
                # for h_candidates and a_candidates
                word = word[0]
            items = word.split()
            new_word = ' '.join(porter.stem(item) for item in items)
            new_words.append(new_word.strip())
        stemmed_sequences.append(new_words)
    
    return stemmed_sequences

In [ ]:
import logging

class Logger(object):

    def __init__(self, filename, level='info'):
        level = logging.INFO if level == 'info' else logging.DEBUG
        self.logger = logging.getLogger(filename)
        self.logger.propagate = False
        self.logger.setLevel(level)  #

        th = logging.FileHandler(filename, 'a')

        self.logger.addHandler(th)

log = Logger('pred/GPT-HA.log')

In [6]:
def calPRF(num_c, num_e, num_s):
    F1 = 0.0
    P = float(num_c) / float(num_e) if num_e!=0 else 0.0
    R = float(num_c) / float(num_s) if num_s!=0 else 0.0
    if (P + R == 0.0):
        F1 = 0
    else:
        F1 = 2 * P * R / (P + R)
    return P, R, F1

def getPRF(references, predictions, log):
    num_c_5, num_c_10, num_c_15 = 0, 0, 0
    num_e_5, num_e_10, num_e_15 = 0, 0, 0
    num_s = 0
    for i  in range(len(references)):
        reference = references[i]
        prediction = predictions[i]
        j = 0
        for candidate in prediction[:15]:
            if candidate in reference:
                if j<5:
                    num_c_5 += 1
                    num_c_10 += 1
                    num_c_15 += 1
                elif (j<10 and j>=5):
                    num_c_10 += 1
                    num_c_15 += 1
                elif (j<15 and j>=10):
                    num_c_15 += 1
            j += 1
        
        if len(prediction[0:5]) == 5:
            num_e_5 += 5
        else:
            num_e_5 += len(prediction[0:5])
        
        if len(prediction[0:10]) == 10:
            num_e_10 += 10
        else:
            num_e_10 += len(prediction[0:10])
        
        if len(prediction[0:15]) == 15:
            num_e_15 += 15
        else:
            num_e_15 += len(prediction[0:15])

        num_s += len(reference)
    
    P, R, F1 = calPRF(num_c_5, num_e_5, num_s)
    log.logger.info("P@5:{} R@5:{} F1@5:{}".format(P,R,F1))
    P, R, F1 = calPRF(num_c_10, num_e_10, num_s)
    log.logger.info("P@10:{} R@10:{} F1@10:{}".format(P,R,F1))
    P, R, F1 = calPRF(num_c_15, num_e_15, num_s)
    log.logger.info("P@15:{} R@15:{} F1@15:{}".format(P,R,F1))

In [22]:
gold_standards_stem = stemmer(keywords)
pred_candidates_stem = stemmer(pred_keywords)

In [23]:
log.logger.info('GPT-AH')
getPRF(gold_standards_stem, pred_candidates_stem, log)